# Hito 1 - Definicion del problema, seleccion de variables y arquitectura de datos

Proyecto: **Music Mood Activity Recommender**

Este notebook documenta las bases del proyecto: el problema de Machine Learning, las variables seleccionadas, las fuentes de datos utilizadas y la arquitectura de ingesta, almacenamiento y procesamiento que soporta el resto del sistema.


## 1. Definicion del problema de Machine Learning

El problema principal del proyecto consiste en construir un sistema capaz de recomendar canciones de Spotify en funcion de dos entradas del usuario:

- como se siente emocionalmente;
- que actividad o situacion va a realizar.

Para poder recomendar canciones de forma contextual, el proyecto se divide en dos problemas relacionados:

1. **Clasificacion emocional de canciones**: predecir la emocion asociada a una cancion a partir de sus caracteristicas acusticas.
2. **Ranking/recomendacion contextual**: ordenar canciones segun la compatibilidad entre emocion del usuario, actividad indicada y caracteristicas musicales.

El primer problema es el nucleo supervisado del proyecto. El segundo utiliza las predicciones emocionales y reglas de supervision debil para construir un recomendador.


In [ ]:
import pandas as pd

example_requests = pd.DataFrame(
    [
        {"user_mood": "sad", "activity": "workout"},
        {"user_mood": "happy", "activity": "study"},
        {"user_mood": "energetic", "activity": "dance"},
        {"user_mood": "calm", "activity": "sleep"},
    ]
)
example_requests

## 2. Tipo de problema

**Tipo principal:** clasificacion supervisada multiclase.

El modelo aprende a clasificar canciones en una de cuatro emociones:

| Etiqueta numerica | Emocion |
|---:|---|
| 0 | sad |
| 1 | happy |
| 2 | energetic |
| 3 | calm |

Posteriormente, esas predicciones se utilizan en una capa de recomendacion que ajusta las canciones al contexto del usuario.


In [4]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
mood_path = ROOT / "datasets" / "mood_dataset.csv"

label_map = {0: "sad", 1: "happy", 2: "energetic", 3: "calm"}
print("Label map:", label_map)

if mood_path.exists():
    sample = pd.read_csv(mood_path, usecols=["labels"], nrows=20000)
    counts = sample["labels"].value_counts().sort_index()
    counts.rename(index=label_map)
else:
    print("mood_dataset.csv not found:", mood_path)

Label map: {0: 'sad', 1: 'happy', 2: 'energetic', 3: 'calm'}


## 3. Contexto y utilidad real

Las plataformas musicales suelen recomendar canciones segun historial, popularidad, genero o artistas similares. Sin embargo, una recomendacion puede ser mas util si tambien considera el estado emocional y la actividad actual del usuario.

Ejemplos de uso real:

- una persona triste que quiere entrenar no necesita necesariamente musica triste, sino canciones emocionalmente compatibles pero con energia suficiente;
- una persona feliz que va a estudiar puede necesitar musica positiva pero no demasiado intensa;
- una persona tranquila que quiere dormir necesita canciones con baja energia y alta calma;
- una persona que quiere desahogarse puede preferir canciones con una emocion triste o calmada.

La utilidad del proyecto esta en combinar informacion musical objetiva, emocion estimada y contexto de actividad para producir recomendaciones mas personalizadas.


In [5]:
import pandas as pd

activity_targets = {
    "study": {"energy_min": 0.2, "energy_max": 0.6, "valence_min": 0.4},
    "workout": {"energy_min": 0.7, "energy_max": 1.0, "valence_min": 0.5},
    "sleep": {"energy_min": 0.0, "energy_max": 0.3, "valence_min": 0.3},
    "relax": {"energy_min": 0.1, "energy_max": 0.4, "valence_min": 0.3},
}

pd.DataFrame.from_dict(activity_targets, orient="index")

,energy_min,energy_max,valence_min
study,0.2,0.6,0.4
workout,0.7,1.0,0.5
sleep,0.0,0.3,0.3
relax,0.1,0.4,0.3


## 4. Fuentes de datos utilizadas

El proyecto integra tres fuentes con estructuras diferentes:

| Dataset | Archivo | Funcion en el proyecto |
|---|---|---|
| Dataset emocional etiquetado | `mood_dataset.csv` | Entrenar el clasificador emocional supervisado |
| Dataset de tracks de Spotify | `spotify_tracks_dataset.csv` | Catalogo inicial de canciones para clasificar y recomendar |
| Dataset de canciones con lyrics | `songs_with_attributes_and_lyrics.csv` | Ampliar catalogo y aportar senales textuales derivadas de letras |

Los datasets no se cargan completos manualmente en memoria durante la exploracion inicial porque algunos tienen gran tamano. La ingesta se realiza por chunks y mediante el pipeline Kafka/Spark.


In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
datasets_dir = ROOT / "datasets"

files = {
    "mood_dataset": datasets_dir / "mood_dataset.csv",
    "spotify_tracks_dataset": datasets_dir / "spotify_tracks_dataset.csv",
    "songs_with_attributes_and_lyrics": datasets_dir / "songs_with_attributes_and_lyrics.csv",
}

for name, path in files.items():
    print(f"\n{name}: {path.name}")
    if path.exists():
        df = pd.read_csv(path, nrows=3)
        print(df.head(3))
    else:
        print("missing:", path)


mood_dataset: mood_dataset.csv
   Unnamed: 0.1  Unnamed: 0  duration (ms)  danceability  energy  loudness  \
0             0           0       195000.0         0.611   0.614    -8.815   
1             1           1       194641.0         0.638   0.781    -6.848   
2             2           2       217573.0         0.560   0.810    -8.029   

   speechiness  acousticness  instrumentalness  liveness  valence    tempo  \
0       0.0672        0.0169          0.000794     0.753    0.520  128.050   
1       0.0285        0.0118          0.009530     0.349    0.250  122.985   
2       0.0872        0.0071          0.000008     0.241    0.247  170.044   

      spec_rate  labels                                   uri  
0  3.446154e-07       2  spotify:track:3v6sBj3swihU8pXQQHhDZo  
1  1.464234e-07       1  spotify:track:7KCWmFdw0TzoJbKtqRRzJO  
2  4.007850e-07       1  spotify:track:2CY92qejUrhyPUASawNVRr  

spotify_tracks_dataset: spotify_tracks_dataset.csv
   Unnamed: 0                track

## 5. Variable objetivo y variables independientes

### Variable objetivo

La variable objetivo del modelo supervisado es:

`mood_label`

Representa la emocion de la cancion: `sad`, `happy`, `energetic` o `calm`.

### Variables independientes principales

Las features acusticas seleccionadas son:

| Feature | Justificacion |
|---|---|
| `danceability` | Mide adecuacion para bailar; relevante para actividades con movimiento |
| `energy` | Representa intensidad; clave para diferenciar canciones calmadas y energicas |
| `speechiness` | Detecta presencia de voz hablada; afecta a estudio, concentracion y rap/spoken word |
| `acousticness` | Indica si la cancion es acustica; relacionada con calma, tristeza o relajacion |
| `instrumentalness` | Diferencia canciones vocales e instrumentales; util para estudiar o dormir |
| `liveness` | Detecta sensacion de directo; puede afectar al contexto de escucha |
| `valence` | Mide positividad musical; muy relevante para distinguir emociones positivas/negativas |
| `loudness` | Volumen global; relacionado con energia e intensidad |
| `tempo` | Ritmo/BPM; importante para correr, entrenar o bailar |
| `duration (ms)` | Duracion; util para filtrar canciones y mantener consistencia del catalogo |
| `spec_rate` | Feature derivada disponible en el dataset emocional; aporta informacion complementaria |

En el dataset de lyrics se generan ademas features textuales ligeras, como longitud de letra, numero de palabras y tasas lexicas asociadas a emociones. Estas variables enriquecen el recomendador, pero no sustituyen al modelo acustico principal.


In [12]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
mood_path = ROOT / "datasets" / "mood_dataset.csv"

features = {
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "loudness",
    "tempo",
    "duration (ms)",
    "spec_rate",
}

if mood_path.exists():
    df = pd.read_csv(mood_path, usecols=features, nrows=5000)
    summary = df.describe().T[["mean", "std", "min", "max"]]
    print(summary)
else:
    print("mood_dataset.csv not found:", mood_path)

                          mean           std           min           max
duration (ms)     2.246852e+05  1.137337e+05  32348.000000  3.600000e+06
danceability      5.956576e-01  1.773088e-01      0.000000  9.870000e-01
energy            6.148014e-01  2.276766e-01      0.000175  1.000000e+00
loudness         -8.772260e+00  5.464049e+00    -48.321000  2.624000e+00
speechiness       9.591940e-02  1.129721e-01      0.000000  9.300000e-01
acousticness      3.098170e-01  3.122706e-01      0.000002  9.950000e-01
instrumentalness  1.372242e-01  2.854411e-01      0.000000  1.000000e+00
liveness          1.944930e-01  1.707349e-01      0.023300  9.970000e-01
valence           4.891821e-01  2.580609e-01      0.000000  9.780000e-01
tempo             1.194483e+02  2.942939e+01      0.000000  2.175910e+02
spec_rate         5.122471e-07  7.631379e-07      0.000000  1.042454e-05


## 6. Justificacion de la seleccion de datos

Se utilizan multiples fuentes porque ninguna contiene por si sola toda la informacion necesaria:

- `mood_dataset.csv` contiene etiquetas emocionales, pero no ofrece un catalogo final con nombres de canciones suficientemente util para recomendacion directa.
- `spotify_tracks_dataset.csv` contiene informacion de canciones, artistas, generos y features acusticas, pero no incluye etiquetas emocionales.
- `songs_with_attributes_and_lyrics.csv` amplia el catalogo e introduce letras, lo que permite enriquecer el sistema con senales semanticas derivadas.

La integracion de estas fuentes permite entrenar un modelo supervisado con el dataset etiquetado y aplicar despues ese conocimiento a catalogos mas amplios.


In [4]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

datasets_dir = ROOT / "datasets"

files = {
    "mood_dataset": datasets_dir / "mood_dataset.csv",
    "spotify_tracks_dataset": datasets_dir / "spotify_tracks_dataset.csv",
    "songs_with_attributes_and_lyrics": datasets_dir / "songs_with_attributes_and_lyrics.csv",
}

columns_by_file = {}
missing = []

for name, path in files.items():

    if path.exists():

        print(f"- Cargando columnas de: {path.name}")

        columns = pd.read_csv(path, nrows=0).columns.tolist()

        columns_by_file[name] = set(columns)

        print(f"{name}: {len(columns)} columns found")

    else:
        missing.append(path.name)

target_features = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "loudness",
    "tempo",
    "duration (ms)",
]

if missing:

    print("Missing datasets:")
    for file in missing:
        print("-", file)

else:

    common = set(target_features).intersection(*columns_by_file.values())

    common_sorted = sorted(common)

    print("\nCaracteristicas comunes encontradas:\n")

    for feature in common_sorted:
        print("-", feature)

- Cargando columnas de: mood_dataset.csv
mood_dataset: 15 columns found
- Cargando columnas de: spotify_tracks_dataset.csv
spotify_tracks_dataset: 21 columns found
- Cargando columnas de: songs_with_attributes_and_lyrics.csv
songs_with_attributes_and_lyrics: 17 columns found

Caracteristicas comunes encontradas:

- acousticness
- danceability
- energy
- instrumentalness
- liveness
- loudness
- speechiness
- tempo
- valence


## 7. Arquitectura de ingesta y almacenamiento

La arquitectura sigue un patron de data lake por capas:

```text
CSV datasets
   -> Kafka Producer
   -> Kafka Topics
   -> Kafka Consumer
   -> Bronze JSONL local + S3
   -> Spark Bronze to Silver
   -> Silver Parquet local + S3
   -> Preparacion Gold en notebooks
   -> Gold Parquet local + S3
   -> Glue Data Catalog + Athena
```

Las capas son:

- **Bronze**: datos crudos ingeridos desde Kafka en formato JSONL.
- **Silver**: datos limpios, tipados, normalizados y deduplicados con Spark.
- **Gold**: datasets preparados para entrenamiento, clasificacion y recomendacion.

S3 actua como repositorio unico del data lake y permite que Athena consulte los datos sin moverlos.


## 8. Tecnologias utilizadas y justificacion tecnica del almacenamiento

La arquitectura no se plantea como una suma de tecnologias aisladas, sino como un sistema de almacenamiento por responsabilidades. La decision principal es usar **Amazon S3 como repositorio unico del data lake** y apoyarlo con servicios especializados para ingesta, procesamiento, consulta, trazabilidad y auditoria.

### Criterios de seleccion

Los criterios usados para elegir el sistema de almacenamiento son:

- **Heterogeneidad de fuentes**: los datasets no tienen la misma estructura. Hay datos emocionales etiquetados, metadatos de canciones, caracteristicas acusticas y letras.
- **Conservacion del dato crudo**: es necesario poder volver al origen si cambia una transformacion o una regla de limpieza.
- **Separacion entre almacenamiento y computo**: los datos deben persistir aunque cambie el motor que los procesa.
- **Trazabilidad**: cada ejecucion del pipeline debe poder auditarse.
- **Preparacion para Machine Learning**: la capa final debe estar en un formato eficiente para entrenamiento, validacion y recomendacion.
- **Consulta reproducible**: debe ser posible validar capas con SQL sin mover los datos a otra base.

### Justificacion por tecnologia

| Requisito del hito | Tecnologia usada | Justificacion tecnica |
|---|---|---|
| Repositorio unico de datos | Amazon S3 | Se usa como data lake central porque permite guardar datos heterogeneos por capas Bronze/Silver/Gold sin imponer un esquema rigido desde el inicio. Es mas adecuado que una base relacional como almacenamiento principal para datos crudos, limpios y preparados. |
| Ingesta en tiempo real | Apache Kafka | Simula la llegada continua de canciones/eventos. Desacopla productores y consumidores: el productor publica eventos y el consumer los persiste en Bronze. Esto acerca el proyecto a un escenario real donde entran nuevos tracks o interacciones. |
| Procesamiento y unificacion | Apache Spark / PySpark | Permite limpiar, tipar, deduplicar, filtrar valores fuera de rango y normalizar fuentes distintas. Aunque se ejecute localmente, la logica es escalable a volumenes mayores. |
| ETL/ELT y catalogacion | AWS Glue Data Catalog | Registra las capas Parquet de S3 como tablas externas. Esto hace que el data lake sea descubrible y consultable sin duplicar datasets. |
| Consulta SQL serverless | Amazon Athena | Valida conteos, esquemas y resultados directamente sobre S3. Encaja con ELT porque consulta los Parquet donde ya estan almacenados. |
| Datos estructurados de auditoria | AWS RDS MySQL | Guarda resumenes tabulares de ejecuciones, datasets, capas y estadisticas de columnas. Esta informacion tiene estructura estable y se consulta bien con SQL. |
| Metadatos flexibles | MongoDB Atlas | Guarda detalle granular por archivo y metadatos variables del data lake. Complementa a RDS: Mongo almacena documentos flexibles y RDS almacena resumen relacional. |
| Automatizacion basada en eventos | AWS Lambda | Reacciona a escrituras en S3 para generar auditoria. Evita tener que lanzar manualmente procesos ligeros tras cada escritura relevante. |
| Script unico | `scripts/run_pipeline.py` | Orquesta despliegue, ingesta, procesamiento, persistencia, catalogacion y validaciones desde un unico punto de entrada. |

### Por que S3 es el almacenamiento principal

S3 se elige como almacenamiento principal porque el proyecto necesita conservar el linaje completo del dato:

```text
Bronze: eventos crudos desde Kafka en JSONL
Silver: datos limpios y normalizados en Parquet
Gold: datasets preparados para ML en Parquet
```

Una unica base de datos relacional no seria la mejor opcion para este caso, porque obligaria a modelar desde el principio fuentes con estructuras diferentes y a almacenar datos crudos que todavia no estan normalizados. MongoDB tampoco seria suficiente por si solo, porque el entrenamiento y la validacion analitica se benefician mas de Parquet, Spark y Athena.

Por eso se adopta una arquitectura hibrida: S3 contiene el dato principal y el resto de sistemas cumplen funciones complementarias de procesamiento, consulta, trazabilidad y auditoria.

### Entregable independiente

La justificacion tecnica completa se entrega tambien como documento separado en:

`docs/JUSTIFICACION_SISTEMAS_ALMACENAMIENTO.md`


## 9. Script unico funcional

El script maestro del pipeline es:

`scripts/run_pipeline.py`

Este script realiza las siguientes tareas:

1. Levanta la arquitectura Docker necesaria para Kafka, MongoDB y Spark.
2. Prepara almacenamiento local y S3.
3. Crea topics Kafka.
4. Produce eventos desde los CSV por chunks.
5. Consume eventos y los guarda en Bronze.
6. Ejecuta transformaciones Spark Bronze -> Silver.
7. Configura notificaciones de S3 para activar AWS Lambda al escribir Bronze/Silver/Gold y generar auditoria en S3.
8. Registra metadatos en MongoDB Atlas y AWS RDS MySQL.
9. Registra tablas en AWS Glue Data Catalog.
10. Ejecuta consultas de validacion en Amazon Athena.

Comando principal:

```powershell
.\.venv\Scripts\python.exe scripts\run_pipeline.py
```

Gold se genera posteriormente en `notebooks/preparacion_datos.ipynb`, porque esa fase incluye preparacion especifica para ML, escalado, split train/test y subida de Gold a S3.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_settings

settings = load_settings()
print("Datasets integrados en el pipeline:")
for dataset in settings.datasets:
    print(f"- {dataset.name}: {dataset.path.name} -> topic {dataset.topic}")

print("\nS3 bucket:", settings.bucket_name)
print("Athena database:", settings.athena_database)
print("RDS MySQL host:", settings.rds_host)
print("MongoDB database:", settings.mongo_database)

Datasets integrados en el pipeline:
- mood_dataset: mood_dataset.csv -> topic mood_dataset_events
- tracks_dataset: spotify_tracks_dataset.csv -> topic tracks_dataset_events
- lyrics_dataset: songs_with_attributes_and_lyrics.csv -> topic lyrics_dataset_events

S3 bucket: music-recommender-lake
Athena database: music_recommender_lake
RDS MySQL host: music-recommender.cjaaeyokabhb.us-east-1.rds.amazonaws.com
MongoDB database: music_recommender


## 10. Repositorio unico e integracion de fuentes

Todas las fuentes terminan integradas en un unico data lake en S3:

```text
s3://<bucket>/bronze/mood_dataset/
s3://<bucket>/bronze/tracks_dataset/
s3://<bucket>/bronze/lyrics_dataset/

s3://<bucket>/silver/mood_dataset/
s3://<bucket>/silver/tracks_dataset/
s3://<bucket>/silver/lyrics_dataset/

s3://<bucket>/gold/mood_prepared/
s3://<bucket>/gold/tracks_prepared/
s3://<bucket>/gold/lyrics_prepared/
```

La unificacion logica se produce porque las fuentes comparten variables acusticas compatibles y se normalizan a un esquema comun para clasificacion y recomendacion.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_settings

settings = load_settings()
base = f"s3://{settings.bucket_name}"
datasets = ["mood_dataset", "tracks_dataset", "lyrics_dataset"]

paths = {
    "bronze": [f"{base}/bronze/{name}/" for name in datasets],
    "silver": [f"{base}/silver/{name}/" for name in datasets],
    "gold": [
        f"{base}/gold/{name.replace('_dataset', '_prepared')}/" for name in datasets
    ],
}

paths

## 11. Que se guarda en cada sistema

### Amazon S3

Guarda los datos del data lake:

- Bronze: eventos crudos.
- Silver: datos limpios en Parquet.
- Gold: datasets preparados para ML.
- Athena results: resultados de consultas SQL.

### AWS RDS MySQL

Guarda trazabilidad tabular y estadisticas estructuradas:

- `pipeline_run_summary`: 1 fila por ejecucion (estado, tiempos, config y totales).
- `dataset_run_summary`: 1 fila por dataset y capa en cada run (filas, tamano, columnas, schema hash).
- `column_stats`: 1 fila por columna/dataset/capa/run (tipo, nulls, min/max).

Ejemplo de consulta:

```sql
SELECT * FROM pipeline_run_summary ORDER BY created_at DESC;
```

### MongoDB Atlas

Guarda detalle granular por archivo Parquet (muchos documentos):

- coleccion `data_lake_files`: 1 documento por archivo con `run_id`, `dataset`, `layer`, `file_name`, `row_count`, `size_bytes` y `row_groups`.

Mongo contiene el detalle por archivo, mientras que RDS guarda el resumen y las estadisticas por columna.

### AWS Lambda

La auditoria se activa por notificaciones de S3 cuando se escriben objetos en Bronze (jsonl) y los marcadores `_SUCCESS` en Silver/Gold. La funcion deriva un evento con `layer`, `dataset` y `location` y guarda una copia en `s3://<bucket>/lambda/audit/<layer>/<dataset>/`.

Ejemplo de evento:

```json
{
  "event_type": "s3_object_created",
  "layer": "silver",
  "dataset": "lyrics_dataset",
  "location": "s3://<bucket>/silver/lyrics_dataset/_SUCCESS",
  "created_at": "2026-05-09T10:30:00Z",
  "source": "ObjectCreated:Put"
}
```

### AWS Glue + Athena

Glue registra tablas externas sobre los Parquet de S3. Athena permite validarlas con SQL:

```sql
SELECT COUNT(*) AS rows_count
FROM music_recommender_lake.silver_lyrics_dataset;
```


In [12]:
import pandas as pd

rds_examples = pd.DataFrame(
    [
        {
            "table": "pipeline_run_summary",
            "key_fields": "run_id, status, started_at, finished_at",
        },
        {
            "table": "dataset_run_summary",
            "key_fields": "run_id, dataset, layer, row_count, size_bytes",
        },
        {
            "table": "column_stats",
            "key_fields": "run_id, dataset, layer, column, nulls, min, max",
        },
    ]
)

mongo_example = {
    "collection": "data_lake_files",
    "document_example": {
        "run_id": "2026-05-09T10:30:00Z",
        "dataset": "lyrics_dataset",
        "layer": "silver",
        "file_name": "part-00001.parquet",
        "row_count": 120000,
        "size_bytes": 5823941,
        "row_groups": 4,
    },
}

print("Ejemplo tablas RDS MySQL")
print(rds_examples)

print("Ejemplo documento mongo")
mongo_example

Ejemplo tablas RDS MySQL
                  table                                       key_fields
0  pipeline_run_summary          run_id, status, started_at, finished_at
1   dataset_run_summary    run_id, dataset, layer, row_count, size_bytes
2          column_stats  run_id, dataset, layer, column, nulls, min, max
Ejemplo documento mongo


{'collection': 'data_lake_files',
 'document_example': {'run_id': '2026-05-09T10:30:00Z',
  'dataset': 'lyrics_dataset',
  'layer': 'silver',
  'file_name': 'part-00001.parquet',
  'row_count': 120000,
  'size_bytes': 5823941,
  'row_groups': 4}}

## 12. Conclusiones del hito

En este hito se define un problema de clasificacion emocional supervisada aplicado a recomendacion musical contextual. Se justifican las variables acusticas y textuales, se integran tres fuentes de datos con estructuras diferentes y se establece una arquitectura completa de ingesta y almacenamiento.

La arquitectura cumple los requisitos principales del enunciado:

- ingesta en tiempo real con Kafka;
- procesamiento con Spark;
- almacenamiento centralizado en S3;
- catalogacion y consulta con Glue/Athena;
- trazabilidad estructurada en AWS RDS MySQL;
- metadatos flexibles en MongoDB Atlas;
- automatizacion/auditoria mediante AWS Lambda activada por eventos de S3;;
- orquestacion desde un script unico funcional.
